In [ ]:
# =========================================================
# --- 0. Import libraries ---
# =========================================================

import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
# =========================================================
# --- 1. Loading CIFAR10 dataset ---
# =========================================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Display a sample image (optional)
plt.imshow(x_train[0])
plt.title(f"Label: {y_train[0]}")
plt.show()

In [ ]:
# =========================================================
# --- 2. Data preprocessing ---
# =========================================================

# Normalize to range [0,1]
x_train = x_train / 255.0
x_test = x_test / 255.0

# One-hot encoding of labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# CIFAR-10 class names (useful for visualization)
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

In [ ]:
# =========================================================
# --- 3. Model ---
# =========================================================

model = Sequential()

# Block 1 learns 32 filters of size 3x3
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)))
model.add(MaxPooling2D((2,2)))

# Block 2 learns 64 filters of size 3x3
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

# Block 3 learns 128 filters of size 3x3
model.add(Conv2D(128, (3,3), activation='relu'))

# Global averaging -> fewer parameters, more stable training
model.add(GlobalAveragePooling2D())

model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))

In [ ]:
# =========================================================
# --- 4. Model compilation ---
# =========================================================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

In [ ]:
# =========================================================
# --- 5. Callbacks ---
# =========================================================

from datetime import datetime

# Timestamp for saving files
ts = datetime.now().strftime("_%Y%m%d_%H%M%S")

base_export_dir = "/content/drive/MyDrive/colab_cifar10_exports"
os.makedirs(base_export_dir, exist_ok=True)

# Stop training if no improvement
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Reduce learning rate if validation stagnates
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

# Save the best model
checkpoint_path = os.path.join(base_export_dir, f"best_model{ts}.keras")

model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
# =========================================================
# --- 6. Training ---
# =========================================================

history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr, model_checkpoint]
)

In [ ]:
# =========================================================
# --- 7. Final evaluation ---
# =========================================================

# Load best model (optional safety)
from tensorflow.keras.models import load_model

if os.path.exists(checkpoint_path):
    model = load_model(checkpoint_path)

loss, acc = model.evaluate(x_test, y_test)
print("Test accuracy:", acc)

In [ ]:
# =========================================================
# --- 8. Training visualization (loss) ---
# =========================================================

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title("Loss")

# Save plot
plot_path = os.path.join(base_export_dir, f"loss_plot{ts}.png")
plt.savefig(plot_path)
plt.show()

In [ ]:
# =========================================================
# --- 9. Misclassification visualization & confusion matrix ---
# =========================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns

# Predictions
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Misclassified indices
mis_idx = np.where(y_pred_classes != y_true)[0]

# Show some incorrect examples
plt.figure(figsize=(10,5))
for i, idx in enumerate(mis_idx[:6]):
    plt.subplot(2,3,i+1)
    plt.imshow(x_test[idx])
    plt.title(f"Pred: {class_names[y_pred_classes[idx]]}\nTrue: {class_names[y_true[idx]]}")
    plt.axis('off')
plt.show()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=False)
plt.title("Confusion Matrix")

# Save plot
cm_path = os.path.join(base_export_dir, f"conf_matrix{ts}.png")
plt.savefig(cm_path)
plt.show()

In [ ]:
# =========================================================
# --- 10. Save model ---
# =========================================================

model_path = os.path.join(base_export_dir, f"my_model{ts}.keras")
model.save(model_path)

print("Model saved as:", model_path)